# SmartRoute-OSM: Algoritmo A* (A-Estrela)

Neste terceiro notebook, carregaremos a malha viária e usaremos o algoritmo **A*** (implementado em `routing/algorithms.py`) com a distância em linha reta (**Fórmula de Haversine**) como heurística geográfica para acelerar a busca do menor caminho.

In [1]:
import os
import sys
import time
import osmnx as ox
import pandas as pd

# Adicionar pasta do projeto ao path para importar módulo de roteamento
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'routing'))
from algorithms import astar

# Carregar o grafo pré-processado
data_path = "../data/quixada_drive.graphml"

if os.path.exists(data_path):
    G = ox.load_graphml(data_path)
    print(f"Grafo carregado com sucesso! Nós: {len(G.nodes)}, Arestas: {len(G.edges)}")
else:
    raise FileNotFoundError("Arquivo 'quixada_drive.graphml' não encontrado. Execute o Notebook 01 primeiro.")

Grafo carregado com sucesso! Nós: 3647, Arestas: 9762


## 1. Definição do Par Origem-Destino
Utilizamos os mesmos pontos do notebook anterior para garantir um comparativo justo entre os algoritmos.

In [2]:
# Mapear coordenadas fixas para os nós
origem_coords = (-4.9685, -39.0161)
destino_coords = (-4.9780, -39.0050)

origem_node = ox.distance.nearest_nodes(G, X=origem_coords[1], Y=origem_coords[0])
destino_node = ox.distance.nearest_nodes(G, X=destino_coords[1], Y=destino_coords[0])

print(f"ID Nó Origem: {origem_node}")
print(f"ID Nó Destino: {destino_node}")

ID Nó Origem: 252615233
ID Nó Destino: 4829461035


## 2. Execução e Medição de Desempenho

In [3]:
start_time = time.time()
astar_path, astar_dist, astar_visited = astar(G, origem_node, destino_node, weight_attribute='length')
execution_time_ms = (time.time() - start_time) * 1000

print(f"=== RESULTADOS A* ===")
print(f"Distância Total: {astar_dist:.2f} metros")
print(f"Nós Visitados: {astar_visited}")
print(f"Tempo de Execução: {execution_time_ms:.2f} ms")

=== RESULTADOS A* ===
Distância Total: 1890.27 metros
Nós Visitados: 81
Tempo de Execução: 4.62 ms


## 5. Salvando Resultados Intermediários

In [4]:
metrics_data = {
    'algorithm': ['A* (Haversine)'],
    'distance_m': [astar_dist],
    'visited_nodes': [astar_visited],
    'execution_time_ms': [execution_time_ms],
    'origem_node': [origem_node],
    'destino_node': [destino_node]
}

df_astar = pd.DataFrame(metrics_data)
df_astar.to_csv("../data/astar_metrics.csv", index=False)

# Salvar o caminho para reusar no mapa
import json
with open("../data/astar_path.json", "w") as f:
    json.dump(astar_path, f)

print("Métricas e caminho salvos na pasta '../data/' com sucesso!")

Métricas e caminho salvos na pasta '../data/' com sucesso!
